In [1]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

torch.set_grad_enabled(False)

In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [3]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

Loaded GPT


In [4]:
kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

# mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)
mixtral = LanguageModel("mistralai/Mistral-7B-Instruct-v0.2", device_map="cuda:0", dispatch=True)

print("Loaded Mixtral in 4bit")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [5]:
import importlib 
import agent
import agent.utils

importlib.reload(agent)
importlib.reload(agent.utils)

env = agent.Environment(mixtral, gpt, sae_list)

In [6]:
layer = 10
index = 6536

prompts = [
    " (getting back 87 cents on the dollar in 2010.) In comparison, the average state gets",
    " (Dungeons and Dragons figurines off the kitchen table).ĊĊThe other day I noteds",
    ", (appears to be in much the same boat.) Yes, our political leaders are perfectly",
]

env(
    prompts = prompts,
    layer = layer,
    index = index
)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Loaded features
Trial 0
[I received a refund of 87 cents out of every dollar spent in the year 2010.]
[Figurines from Dungeons and Dragons were once kept on my kitchen table.]
[Political leaders seem to find themselves in similar financial situations, appearing to be in the same boat with 87 cents returned on every dollar spent in 2010.]
['I received a refund of 87 cents out of every dollar spent in the year 2010.', 'Figurines from Dungeons and Dragons were once kept on my kitchen table.', 'Political leaders seem to find themselves in similar financial situations, appearing to be in the same boat with 87 cents returned on every dollar spent in 2010.']


┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Content                                                                                                  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ User      │ You are a meticulous AI researcher conducting a high-stakes investigation on neurons in a large language │
│           │ model. Your task is to understand what features of the input text cause a specific neuron to activate.   │
│           │                                                                                                          │
│           │ You will be given a list of text samples containing tokens on which the neuron activates strongly. The   │
│           │ specific tokens which caused the neuron to activate strongly will appear between bars like ** this**. If │
│           │ multiple tokens cause the neuron to activate strongly, the entire sequence will be contained between     │
│           │ bars ** just like this**.                                                                                │
│           │                                                                                                          │
│           │ You will be given multiple samples on which a neuron activates strongly. For each sample in turn, note   │
│           │ down a few features that the text possesses, even if you don't initially think they are important.       │
│           │                                                                                                          │
│           │ Once you have written down a few notes for each text sample, summarize what highly-activating samples    │
│           │ have in common. Finally, use your findings to produce a plausible explaination for what causes the       │
│           │ neuron to fire.                                                                                          │
│           │                                                                                                          │
│           │ Sample 0:<|endoftext|> (getting back 87 cents on the** dollar** in** 2010**.) In comparison, the average │
│           │ state gets                                                                                               │
│           │ Sample 1:<|endoftext|> (Dungeons and Dragons figurines off the** kitchen**** table**).����The other day  │
│           │ I noteds                                                                                                 │
│           │ Sample 2:<|endoftext|>, (appears to be in much the same** boat**.) Yes, our political leaders are        │
│           │ perfectly                                                                                                │
│           │                                                                                                          │
│ Assistant │ Sample 0: This text contains the phrases "getting back 87 cents on the dollar" and "in 2010." The neuron │
│           │ seems to respond to currency-related expressions and specific years.                                     │
│           │                                                                                                          │
│           │ Sample 1: The text mentions "kitchen table" and "Dungeons and Dragons figurines." The neuron may react   │
│           │ to objects being placed on tables or references to games or hobbies.                                     │
│           │                                                                                                          │
│           │ Sample 2: The phrase "appears to be in much the same boat" was highlighted. The neuron might respond to  │
│           │ idioms or expressions related to similar situations.                                                     │
│      

Trial 1


KeyboardInterrupt: 